## PRACTICA OBLIGATORIA: **Transfer Learning y Fine Tuning con CNN**

---
### SOLUCIÓN

> Este notebook resuelve el ejercicio propuesto reutilizando el dataset de clasificación de **paisajes** de la práctica de la unidad anterior (imágenes organizadas en carpetas, una por clase: por ejemplo `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`).
>
> **Importante:** este notebook está pensado para ejecutarse **en local**, ya que el entrenamiento de los modelos preentrenados consume bastantes recursos. Ajusta las rutas de la sección *0. Configuración* a la ubicación real de tus datos.


### Ejercicio 0

Importa los paquetes y módulos que necesites a lo largo del notebook.

In [ ]:
# --- Librerías generales ---
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Sklearn: métricas y utilidades ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils import shuffle

# --- OpenCV para lectura/redimensionado de imágenes ---
import cv2

# --- TensorFlow / Keras ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import MobileNetV2, VGG19, InceptionV3, ResNet50V2

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs disponibles:", tf.config.list_physical_devices('GPU'))


### Objetivo del ejercicio

Comparar una red convolucional hecha ad-hoc frente a los modelos preentrenados y ajustados con fine tuning y transfer learning. Para ello emplea el dataset de paisajes del conjunto de ejercicios de la unidad anterior.


### Se pide

1. Preparar los datos del modelo y las funciones de visualización, copia para ello todo lo que necesites de las soluciones del ejercicio de clasificación de paisajes de la unidad anterior.

2. Escoger entre uno de los modelos VGG-19, InceptionV3 y MobileNetV2 (todos en https://keras.io/api/applications/) (Se aconseja este último si no tenemos un ordenador muy potente). Si no te haces con estos puedes recurrir a la ResNetV50.

4. Hacer un transfer-learning con una cabeza de como mucho 2 capas densas ocultas y una de salida. Mostrar la evaluación contra test, el report de clasificación y la matriz de confusión.

5. Hacer un fine-tuning con la misma cabeza diseñada en el punto anterior. Mostrar la evaluación contra test, el report de clasificación y la matriz de confusión.

6. Comparar los resultados con los obtenidos con la red convolucional del ejercicio mencionado.

EXTRA:
- Repetir el transfer learning empleando aumentado de imágenes.


## 0. Configuración de rutas y parámetros

Adapta estas rutas a la estructura real de tu dataset de paisajes (la misma que usaste en la práctica de la unidad anterior). Se asume la estructura típica de este tipo de dataset:

```
data/
  seg_train/
      buildings/
      forest/
      glacier/
      mountain/
      sea/
      street/
  seg_test/
      buildings/
      forest/
      glacier/
      mountain/
      sea/
      street/
```

Si tu estructura es distinta (por ejemplo un único directorio con todas las imágenes y un dataframe con filename/clase), basta con adaptar la función `cargar_dataset_desde_carpetas` de la sección 1.

In [ ]:
# --- Rutas del dataset (AJUSTAR a tu entorno local) ---
TRAIN_DIR = "./data/seg_train"
TEST_DIR  = "./data/seg_test"

# --- Parámetros de imagen ---
IMG_SIZE   = (150, 150)   # tamaño al que redimensionamos todas las imágenes
BATCH_SIZE = 32
VAL_SPLIT  = 0.2          # % del train que usamos como validación

# Clases: se infieren automáticamente a partir de los nombres de las subcarpetas de TRAIN_DIR
CLASSES = sorted(os.listdir(TRAIN_DIR)) if os.path.isdir(TRAIN_DIR) else []
NUM_CLASSES = len(CLASSES)
print("Clases detectadas:", CLASSES)
print("Número de clases:", NUM_CLASSES)


## 1. Preparación de los datos y funciones de visualización

Se reutilizan (adaptadas) las funciones de la práctica de clasificación de paisajes de la unidad anterior: una función para cargar un subconjunto de imágenes como arrays de NumPy (útil para el EDA y para mostrar ejemplos/matriz de confusión con imágenes), y los generadores de Keras (`ImageDataGenerator`) para el entrenamiento con `flow_from_directory`, que es más eficiente en memoria para datasets grandes de imágenes.

In [ ]:
def cargar_dataset_desde_carpetas(directorio, clases, img_size=IMG_SIZE, n_por_clase=None):
    """
    Recorre un directorio con subcarpetas por clase y devuelve:
        X -> array de imágenes (N, H, W, 3) normalizado [0,1]
        y -> array de etiquetas enteras (N,)

    Pensada para EDA / visualización rápida, no para entrenar directamente
    con datasets muy grandes (para eso usamos los generadores de Keras).

    n_por_clase: si se indica, limita cuántas imágenes se leen por clase
                 (útil para hacer un EDA rápido sin cargar todo el dataset).
    """
    X, y = [], []
    for idx_clase, clase in enumerate(clases):
        ruta_clase = os.path.join(directorio, clase)
        ficheros = glob.glob(os.path.join(ruta_clase, "*"))
        if n_por_clase is not None:
            ficheros = ficheros[:n_por_clase]
        for fichero in ficheros:
            img = cv2.imread(fichero)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, img_size)
            X.append(img)
            y.append(idx_clase)
    X = np.array(X, dtype="float32") / 255.0
    y = np.array(y)
    return X, y


def plot_sample_images(X, y, clases, n_muestras=12, n_columnas=6, titulo=None):
    """Pinta una rejilla de imágenes con su etiqueta de clase."""
    n_muestras = min(n_muestras, len(X))
    n_filas = int(np.ceil(n_muestras / n_columnas))
    idxs = np.random.choice(len(X), n_muestras, replace=False)

    plt.figure(figsize=(n_columnas * 2, n_filas * 2))
    if titulo:
        plt.suptitle(titulo)
    for i, idx in enumerate(idxs):
        plt.subplot(n_filas, n_columnas, i + 1)
        plt.imshow(X[idx])
        plt.title(clases[y[idx]], fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def plot_training_history(history, titulo=""):
    """Curvas de accuracy y loss de entrenamiento/validación."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history["accuracy"], label="train")
    axes[0].plot(history.history["val_accuracy"], label="val")
    axes[0].set_title(f"Accuracy {titulo}")
    axes[0].set_xlabel("Época")
    axes[0].legend()

    axes[1].plot(history.history["loss"], label="train")
    axes[1].plot(history.history["val_loss"], label="val")
    axes[1].set_title(f"Loss {titulo}")
    axes[1].set_xlabel("Época")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(y_true, y_pred, clases, titulo="Matriz de confusión"):
    """Dibuja la matriz de confusión como heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=clases, yticklabels=clases)
    plt.xlabel("Predicción")
    plt.ylabel("Clase real")
    plt.title(titulo)
    plt.tight_layout()
    plt.show()
    return cm


def evaluar_modelo(modelo, generador_test, clases, nombre_modelo="modelo"):
    """
    Evalúa un modelo Keras sobre un generador de test:
    - loss / accuracy
    - classification report
    - matriz de confusión
    Devuelve un diccionario con las métricas para poder comparar modelos después.
    """
    loss, acc = modelo.evaluate(generador_test, verbose=0)
    print(f"\n=== Evaluación en test: {nombre_modelo} ===")
    print(f"Loss: {loss:.4f}  |  Accuracy: {acc:.4f}")

    generador_test.reset()
    y_true = generador_test.classes
    probs = modelo.predict(generador_test, verbose=0)
    y_pred = np.argmax(probs, axis=1)

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=clases))

    plot_confusion_matrix(y_true, y_pred, clases,
                           titulo=f"Matriz de confusión - {nombre_modelo}")

    return {"nombre": nombre_modelo, "loss": loss, "accuracy": acc}


### EDA rápido

Cargamos un pequeño subconjunto de imágenes (no todo el dataset, por memoria) únicamente para visualizar ejemplos y comprobar el balanceo de clases.

In [ ]:
if CLASSES:
    X_eda, y_eda = cargar_dataset_desde_carpetas(TRAIN_DIR, CLASSES, n_por_clase=60)

    print("Shape X_eda:", X_eda.shape)
    plot_sample_images(X_eda, y_eda, CLASSES, n_muestras=12, titulo="Ejemplos del dataset de paisajes")

    # Distribución de clases (mini EDA)
    serie_clases = pd.Series([CLASSES[i] for i in y_eda])
    plt.figure(figsize=(6, 4))
    serie_clases.value_counts().plot(kind="bar")
    plt.title("Distribución de clases (muestra EDA)")
    plt.ylabel("Nº de imágenes")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No se ha encontrado TRAIN_DIR. Ajusta la ruta en la sección 0 antes de continuar.")


### Generadores de datos (Keras `ImageDataGenerator`)

Para el entrenamiento en sí usamos generadores en lugar de cargar todas las imágenes en memoria: es la forma estándar de trabajar con datasets de imágenes de tamaño medio/grande y además nos permite aplicar aumentado de datos de forma sencilla en la parte EXTRA.

- **Train**: se reserva un `VAL_SPLIT` para validación mediante `validation_split`.
- **Test**: solo reescalado (nunca aumentado en test).

In [ ]:
# Generador base (sin aumentado) para el transfer learning y el fine-tuning
train_datagen_base = ImageDataGenerator(rescale=1./255, validation_split=VAL_SPLIT)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen_base.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=SEED,
)

val_generator = train_datagen_base.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

# Nos aseguramos de que el orden de clases coincide con el que usaremos en los reports
CLASSES = list(train_generator.class_indices.keys())
NUM_CLASSES = len(CLASSES)
print("Clases (orden generador):", CLASSES)


## 2. Elección del modelo preentrenado

Se elige **MobileNetV2**, tal y como se aconseja en el enunciado por ser un modelo ligero y rápido de entrenar en local. La arquitectura y filosofía de trabajo (`include_top=False`, congelar capas base, añadir cabeza propia) es la misma para VGG-19, InceptionV3 o ResNet50V2; basta con cambiar la clase importada y, si procede, el tamaño de entrada recomendado por cada modelo.

> Si tu equipo lo permite, puedes cambiar `MODEL_NAME` por `"VGG19"`, `"InceptionV3"` o `"ResNet50V2"` y la celda siguiente construirá el modelo base correspondiente sin tener que tocar el resto del notebook.

In [ ]:
MODEL_NAME = "MobileNetV2"   # Opciones: "MobileNetV2", "VGG19", "InceptionV3", "ResNet50V2"

def construir_modelo_base(nombre, input_shape):
    if nombre == "MobileNetV2":
        return MobileNetV2(weights="imagenet", include_top=False, input_shape=input_shape)
    elif nombre == "VGG19":
        return VGG19(weights="imagenet", include_top=False, input_shape=input_shape)
    elif nombre == "InceptionV3":
        return InceptionV3(weights="imagenet", include_top=False, input_shape=input_shape)
    elif nombre == "ResNet50V2":
        return ResNet50V2(weights="imagenet", include_top=False, input_shape=input_shape)
    else:
        raise ValueError(f"Modelo no soportado: {nombre}")


input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)
modelo_base = construir_modelo_base(MODEL_NAME, input_shape)

# --- Congelamos TODAS las capas del modelo base: transfer learning "clásico" ---
modelo_base.trainable = False

print(f"Modelo base: {MODEL_NAME}")
print("Nº de capas del modelo base:", len(modelo_base.layers))
print("Parámetros del modelo base (congelados):", modelo_base.count_params())


## 3. Transfer Learning: cabeza de clasificación

Construimos la cabeza que sustituye a las capas densas originales del modelo preentrenado, respetando el límite de **2 capas densas ocultas + 1 capa de salida**:

- `GlobalAveragePooling2D` para pasar del mapa de features 3D a un vector (alternativa más ligera que `Flatten`, muy habitual en transfer learning con estas arquitecturas).
- `Dense(256, relu)` + `Dropout` — 1ª capa oculta.
- `Dense(128, relu)` + `Dropout` — 2ª capa oculta.
- `Dense(NUM_CLASSES, softmax)` — capa de salida (clasificación multiclase).

In [ ]:
def construir_cabeza(modelo_base, num_clases):
    x = modelo_base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    salida = layers.Dense(num_clases, activation="softmax")(x)
    return models.Model(inputs=modelo_base.input, outputs=salida)


modelo_tl = construir_cabeza(modelo_base, NUM_CLASSES)

modelo_tl.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

modelo_tl.summary()


In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

EPOCHS_TL = 15

history_tl = modelo_tl.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_TL,
    callbacks=[early_stopping],
)


In [ ]:
plot_training_history(history_tl, titulo="- Transfer Learning")

resultados_tl = evaluar_modelo(modelo_tl, test_generator, CLASSES, nombre_modelo="Transfer Learning (base congelada)")


## 4. Fine-Tuning

Partimos del modelo anterior (misma cabeza) y descongelamos las **últimas capas** del modelo base para que se ajusten también a nuestro dataset. Es importante:

- Bajar el `learning_rate` (respecto al transfer learning) para no destruir los pesos preentrenados con gradientes grandes.
- Recompilar el modelo tras cambiar `trainable`, ya que Keras "congela" la configuración en el momento de la compilación.

In [ ]:
# Descongelamos el modelo base y volvemos a congelar todas las capas
# salvo el último "bloque" (aproximadamente el último 20% de capas).
modelo_base.trainable = True

N_CAPAS_A_DESCONGELAR = max(1, int(len(modelo_base.layers) * 0.2))
capas_congeladas = modelo_base.layers[:-N_CAPAS_A_DESCONGELAR]
capas_descongeladas = modelo_base.layers[-N_CAPAS_A_DESCONGELAR:]

for capa in capas_congeladas:
    capa.trainable = False
for capa in capas_descongeladas:
    capa.trainable = True

print(f"Capas descongeladas para fine-tuning: {len(capas_descongeladas)} de {len(modelo_base.layers)}")

# Recompilamos con un learning rate mucho más bajo
modelo_tl.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

modelo_tl.summary()


In [ ]:
EPOCHS_FT = 15

early_stopping_ft = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

history_ft = modelo_tl.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FT,
    callbacks=[early_stopping_ft],
)


In [ ]:
plot_training_history(history_ft, titulo="- Fine Tuning")

resultados_ft = evaluar_modelo(modelo_tl, test_generator, CLASSES, nombre_modelo="Fine Tuning")


## 5. Comparativa con la CNN ad-hoc de la unidad anterior

Comparamos los resultados obtenidos aquí con los de la red convolucional construida "desde cero" (sin transfer learning) en la práctica de la unidad anterior. Sustituye `ACCURACY_CNN_ADHOC` y `LOSS_CNN_ADHOC` por los valores de test que obtuviste en aquel notebook.

In [ ]:
# --- Resultado de la CNN ad-hoc de la unidad anterior (ajusta estos valores) ---
ACCURACY_CNN_ADHOC = None   # p.ej. 0.72
LOSS_CNN_ADHOC = None       # p.ej. 0.85

resultados_comparativa = [
    {"nombre": "CNN ad-hoc (unidad anterior)", "loss": LOSS_CNN_ADHOC, "accuracy": ACCURACY_CNN_ADHOC},
    resultados_tl,
    resultados_ft,
]

df_comparativa = pd.DataFrame(resultados_comparativa)
display(df_comparativa)

plt.figure(figsize=(6, 4))
plt.bar(df_comparativa["nombre"], df_comparativa["accuracy"])
plt.ylabel("Accuracy en test")
plt.title("Comparativa de modelos")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


**Conclusiones esperables** (complétalas con tus resultados reales):

- El modelo ad-hoc, al entrenarse desde cero solo con los datos propios (muchas menos imágenes que ImageNet), suele obtener peor accuracy y necesitar más épocas para converger.
- El **transfer learning** (base congelada) parte ya de features visuales muy potentes aprendidas en ImageNet y normalmente mejora bastante al ad-hoc con poco entrenamiento.
- El **fine-tuning** suele mejorar todavía más al transfer learning puro porque las últimas capas se ajustan a las particularidades del dataset de paisajes, a costa de más tiempo de entrenamiento y mayor riesgo de sobreajuste si hay pocos datos.

## EXTRA: Transfer Learning con aumentado de imágenes (Image Augmentation)

Repetimos el transfer learning (base congelada, misma cabeza) pero usando un `ImageDataGenerator` con aumentado de datos en el conjunto de entrenamiento: rotaciones, desplazamientos, zoom y flip horizontal. El conjunto de validación y test **nunca se aumentan**, solo se reescalan.

In [ ]:
train_datagen_aug = ImageDataGenerator(
    rescale=1./255,
    validation_split=VAL_SPLIT,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
)

train_generator_aug = train_datagen_aug.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=SEED,
)

val_generator_aug = train_datagen_aug.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
)

# Modelo nuevo: mismo modelo base preentrenado (congelado) + misma cabeza
modelo_base_aug = construir_modelo_base(MODEL_NAME, input_shape)
modelo_base_aug.trainable = False

modelo_tl_aug = construir_cabeza(modelo_base_aug, NUM_CLASSES)
modelo_tl_aug.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

early_stopping_aug = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)

history_tl_aug = modelo_tl_aug.fit(
    train_generator_aug,
    validation_data=val_generator_aug,
    epochs=EPOCHS_TL,
    callbacks=[early_stopping_aug],
)


In [ ]:
plot_training_history(history_tl_aug, titulo="- Transfer Learning con Data Augmentation")

resultados_tl_aug = evaluar_modelo(modelo_tl_aug, test_generator, CLASSES,
                                    nombre_modelo="Transfer Learning + Augmentation")

# Comparativa final incluyendo la versión con aumentado
df_comparativa_final = pd.DataFrame([
    {"nombre": "CNN ad-hoc (unidad anterior)", "loss": LOSS_CNN_ADHOC, "accuracy": ACCURACY_CNN_ADHOC},
    resultados_tl,
    resultados_ft,
    resultados_tl_aug,
])
display(df_comparativa_final)
